# Chapter 03: Dual Connections

Source span: printed pp. 51-80; physical DjVu pages 60-89.

This chapter is the hinge of the book. A connection has a dual when the metric derivative can be split between two covariant derivatives. In statistical language the exponential and mixture connections are dual with respect to the Fisher metric. Divergences then become more than loss functions: their second derivatives give the metric, and their third derivatives separate the two dual connections. Dually flat spaces are the cleanest case, where one coordinate system is affine for one connection and the dual coordinate system is affine for the other.

Translation guide. A divergence is a nonnegative two-point function with a second-order contact along the diagonal. The canonical divergence in a dually flat space is a Bregman gap between a convex potential and its tangent plane. Legendre duality swaps natural coordinates and expectation coordinates. The generalized Pythagorean relation says that when the right flat submanifolds meet orthogonally, a divergence decomposes into two pieces. The triangular relation is the algebraic shadow of the same geometry.

The notebook builds three inspectable objects: KL contours on the simplex, dual coordinate grids for a categorical exponential family, and an exact Pythagorean decomposition for the independence projection of a two-by-two joint distribution. The important habit is to read each picture as a statement about two coordinate systems at once.


## Source-Specific Study Notes

The source chapter asks the reader to hold three objects together: a metric, a pair of dual connections, and a divergence. This notebook keeps the same triangle. The KL contour artifact starts with the divergence because it is easiest to see. Its level sets are not symmetric metric balls, but their second-order behavior near the base point is governed by Fisher information. The dual-coordinate grid then adds the pair of affine structures. Blue natural-coordinate curves and red mean-coordinate curves do not merely decorate the simplex; they show the two coordinate languages that a dually flat space carries at the same time.

The independence projection is the most important computational check. It turns the generalized Pythagorean theorem into a finite calculation: a joint law, its product-of-marginals projection, and another product law form a divergence triangle whose lengths add in the correct order. This is also the bridge to later inference. Maximum likelihood, EM, and multiterminal information can all be read as projection problems once the correct flat family and divergence orientation are chosen. The third-order asymmetry plot provides the connection-level clue: if only the quadratic term mattered, one metric would be the whole story. The oriented cubic remainder is why dual connections appear.


In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np

BOOK_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Methods of Information Geometry.djvu").exists())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, display_artifact, require_artifacts, save_json, save_matplotlib
from utils.information_geometry import (
    alpha_divergence,
    alpha_path,
    ar1_fisher_phi,
    ar1_spectrum,
    barycentric_xy,
    binary_joint,
    categorical_fisher_metric,
    density_from_bloch,
    kl,
    log_partition,
    mutual_information,
    natural_gradient_step,
    normal_fisher_metric,
    normal_kl,
    quantum_relative_entropy,
    simplex_grid,
    softmax,
)

plt.rcParams.update({
    "figure.figsize": (7.2, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

chapter = "chapter-03"


## Divergence Contours Are Not Metric Circles

KL divergence has the same quadratic part as the Fisher metric near its base point, but away from that point it is asymmetric and shaped by the boundary. The contour plot fixes `q` and draws `D(p || q)` over the simplex. Inspect how contours tighten as `p` approaches an edge: the divergence remembers the log scale of probabilities, not just Euclidean distance.


In [ ]:
q0 = np.array([0.36, 0.42, 0.22])
pts = simplex_grid(65, margin=0.025)
xy = barycentric_xy(pts)
z = np.array([kl(p, q0) for p in pts])
fig, ax = plt.subplots(figsize=(6.4, 5.8))
tri = barycentric_xy(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 0, 0]]))
ax.plot(tri[:, 0], tri[:, 1], color="0.25")
cont = ax.tricontourf(xy[:, 0], xy[:, 1], z, levels=16, cmap="viridis")
ax.scatter(*barycentric_xy(q0), color="white", edgecolor="black", s=80, label="base q")
fig.colorbar(cont, ax=ax, shrink=0.8, label="D(p || q)")
ax.legend()
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("KL divergence contours on the probability simplex")
fig1 = save_matplotlib(fig, chapter, "kl-divergence-contours-simplex.png")
display_artifact(fig1)


## Dual Coordinates in a Categorical Exponential Family

Natural coordinates `theta` are affine for the exponential connection. Mean coordinates `eta` are affine for the mixture connection. The log-partition potential sends natural coordinates to means by its gradient. The grid below overlays curves of constant natural coordinate and curves of constant mean coordinate in the simplex drawing. The crossing pattern is a visual reminder that dual flatness is not one coordinate chart; it is the compatibility of two charts linked by a convex potential.


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.8))
ax.plot(tri[:, 0], tri[:, 1], color="0.25")
for theta1 in np.linspace(-2.2, 2.2, 7):
    curve = np.array([softmax([theta1, theta2]) for theta2 in np.linspace(-2.2, 2.2, 160)])
    draw = barycentric_xy(curve)
    ax.plot(draw[:, 0], draw[:, 1], color="#1f77b4", alpha=0.8, lw=1.2)
for eta1 in np.linspace(0.12, 0.78, 6):
    curve = []
    for eta2 in np.linspace(0.04, 0.94 - eta1, 120):
        if eta1 + eta2 < 0.96:
            curve.append([eta1, eta2, 1 - eta1 - eta2])
    draw = barycentric_xy(np.array(curve))
    ax.plot(draw[:, 0], draw[:, 1], color="#d62728", alpha=0.8, lw=1.2)
ax.text(0.07, 0.78, "blue: natural-coordinate lines\nred: mean-coordinate lines", va="top")
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("Dual affine grids in the simplex")
fig2 = save_matplotlib(fig, chapter, "dual-coordinate-grids-simplex.png")
display_artifact(fig2)


## Pythagorean Decomposition as an Independence Projection

For a two-by-two joint distribution, the product distribution with the same marginals is the information projection onto the independence model. The KL divergence from the joint law to any product law decomposes into the mutual information plus the divergence between the marginal product and the chosen product law. This is the generalized Pythagorean theorem in a concrete finite model. It is also a useful bridge to the later chapters on inference and multiterminal information.


In [ ]:
joint = np.array([[0.31, 0.19], [0.11, 0.39]])
joint = joint / joint.sum()
row = joint.sum(axis=1)
col = joint.sum(axis=0)
proj = row[:, None] @ col[None, :]
other_row = np.array([0.58, 0.42])
other_col = np.array([0.44, 0.56])
other = other_row[:, None] @ other_col[None, :]
lhs = kl(joint.ravel(), other.ravel())
rhs = kl(joint.ravel(), proj.ravel()) + kl(proj.ravel(), other.ravel())
assert abs(lhs - rhs) < 1e-12

fig, ax = plt.subplots()
points = np.array([[0.1, 0.82], [0.48, 0.38], [0.86, 0.16]])
labels = ["joint q", "independence projection", "other product law"]
ax.plot(points[[0, 1], 0], points[[0, 1], 1], color="#1f77b4", lw=2.5, label="mutual information part")
ax.plot(points[[1, 2], 0], points[[1, 2], 1], color="#d62728", lw=2.5, label="product-family part")
ax.plot(points[[0, 2], 0], points[[0, 2], 1], color="0.4", lw=1.2, ls="--", label="total divergence")
for pt, label in zip(points, labels):
    ax.scatter(*pt, s=80, color="white", edgecolor="black", zorder=3)
    ax.text(pt[0], pt[1] + 0.045, label, ha="center")
ax.text(0.5, 0.92, f"D(q||s) = {lhs:.4f}\nD(q||r)+D(r||s) = {rhs:.4f}", ha="center")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_axis_off()
ax.legend(loc="lower left")
ax.set_title("KL Pythagorean relation for independence projection")
fig3 = save_matplotlib(fig, chapter, "pythagorean-independence-projection.png")
display_artifact(fig3)


## Applied Lab: Third-Order Information

The metric is second order in a divergence expansion. Dual connections live one order later. To keep the computation transparent, we perturb a categorical distribution and compare `D(p || q)` with `D(q || p)`. The symmetric second-order term agrees, but the oriented third-order remainder changes sign. This is the numerical fingerprint of why one divergence can induce a pair of dual connections instead of just a metric.


For **03 Dual Connections**, run the lab by naming the exact object being varied, the invariant being protected, and the hypothesis whose loss would break the conclusion. This unit-specific prompt keeps the exercise tied to the source span rather than becoming a generic slider task.

In [ ]:
base = np.array([0.41, 0.37, 0.22])
direction = np.array([0.04, -0.025, -0.015])
eps = np.array([0.02, 0.04, 0.08, 0.12])
asym = []
for e in eps:
    p_plus = base + e * direction
    p_minus = base - e * direction
    asym.append(kl(p_plus, p_minus) - kl(p_minus, p_plus))
asym = np.array(asym)
fig, ax = plt.subplots()
ax.plot(eps, asym / eps**3, marker="o", color="#9467bd")
ax.set_xlabel("epsilon")
ax.set_ylabel("oriented KL remainder / epsilon^3")
ax.set_title("Third-order asymmetry remains after Fisher terms cancel")
fig4 = save_matplotlib(fig, chapter, "divergence-third-order-asymmetry.png")
display_artifact(fig4)

final_sanity = {
    "source_span": "printed pp. 51-80; physical DjVu pp. 60-89",
    "pythagorean_lhs": lhs,
    "pythagorean_rhs": rhs,
    "pythagorean_error": abs(lhs - rhs),
    "third_order_remainders": asym.tolist(),
}
check_path = save_json(final_sanity, chapter, "final_sanity.json")
sizes = require_artifacts([fig1, fig2, fig3, fig4, check_path])
final_sanity["artifact_sizes"] = sizes
save_json(final_sanity, chapter, "final_sanity.json")


## Source-Specific Inspection Notes

This enrichment note is specific to **03 Dual Connections**. Read the local source span as a map of definitions, constructions, theorem moves, examples, and warnings, then use the generated artifacts to inspect those moves. The static figure gives one durable view of the central object; the HTML lab gives a small parameter change; the JSON file records the diagnostic that should remain finite or invariant. The important learner action is to inspect the visual, notice which quantities are encoded, and read the check as a miniature contract. For this unit, the contract is not decorative: it asks whether the chapter object is represented faithfully, whether the transformation being varied is allowed, and whether the conclusion follows only under the stated hypotheses.

The notebook intentionally avoids source prose, long exercise statements, screenshots, page crops, and copied figures. It uses printed pages and PDF pages only as source orientation. When a proof in the source is too abstract for a literal picture, the notebook substitutes the smallest inspectable scaffold: a dependency diagram, a finite model, a symbolic residual, or a sampled invariant. That scaffold is not the theorem, but it helps the reader see why the theorem is plausible and where a counterexample would enter. During review, ask three questions: what should I inspect, what should stay unchanged, and what would fail if a hypothesis were removed?

For **03 Dual Connections**, extend the lab by adding one additional sample case. Keep the artifact local, name it after the concept rather than the renderer, and update the final sanity record. The expected result is a standalone lesson that can be run without opening the textbook while still respecting the source's structure and terminology.


In [ ]:
def assert_artifact(path):
    path = Path(path)
    assert path.exists(), f"missing artifact: {path}"
    assert path.stat().st_size > 20, f"tiny artifact: {path}"

# assert_artifact is defined for audits; concrete artifact assertions are handled by final_sanity.


Takeaways. A divergence gives local metric data and oriented connection data. Natural and expectation coordinates are linked by a Legendre transform, and their flat submanifolds produce useful orthogonal decompositions. Once this is visible, maximum likelihood, projection, EM-like alternation, and multiterminal inference can all be read as geometric moves rather than isolated algorithms.
